# Trabajo práctico integrador - Grupo 4
Este es el proceso de minería de datos para obtener el sistema recomendador utilizado por el sistema. Para la realización del proceso, utilizaremos la metodología CRISP-DM. Seguiremos sus fases y actividades genéricas siempre que consideremos que sean requeridas, y en caso de saltear alguna justificaremos por qué no la incluimos.

## Fase 1 - Entendimiento del negocio
Nuestro objetivo en esta fase es entender el problema de negocio y traducirlo a un problema de minería de datos. 

### Actividad 1.1: Determinar objetivos de negocio
El objetivo de negocio es aumentar las ganancias por ventas de videojuegos en un 10% para el año 2026 (comparandolo con las ventas del 2025).

### Actividad 1.2: Evaluar la situación actual
El caso a tratar es sobre una plataforma virtual de venta de videojuegos. Contamos con la siguiente información del sistema:
* Hay un total de 100 juegos en venta.
* El sistema actual de recomendación muestra artículos aleatorios.
* No se cuenta con una base de datos (no hay usuarios, ni preferencias, ni ítems).

Y como supuestos del problema:
* No se agregarán nuevos ítems al carrusel.
* Cada usuario suele comprar entre 7 y 10 juegos.

### Actividad 1.3: Determinar los objetivos de la minería de datos
El objetivo de la minería de datos es crear un sistema recomendador capaz de recomendar k videojuegos a un usuario determinado, de tal forma que las recomendaciones cumplan con los siguientes requisitos:
* Novedad: El sistema recomendador no debe recomendar videojuegos ya puntuados por el usuario.
* Diversidad: La diversidad de las recomendaciones generadas por el sistema recomendador no deberá ser inferior en más de un 20% a la diversidad obtenida mediante un sistema de recomendaciones aleatorias.
* Serendipia: Al menos un 15% de los artículos recomendados deben ser inesperados para el usuario.
* Relevancia: Al menos un 60% de los artículos recomendados deberán ser relevantes para el usuario.

*NOTA PARA EL FUTURO, BORRAR DESPUÉS*

PARA VERIFICAR LA SERENDIPIA: Se considerarán recomendaciones esperables aquellas que coincidan con las sugerencias obtenidas mediante un modelo trivial de recomendación (por ejemplo, basado en popularidad o similitud directa con artículos consumidos previamente por ese usuario). Las recomendaciones que resulten relevantes para el usuario pero no pertenezcan a ese conjunto serán consideradas serendípicas.

PARA VERIFICAR LA RELEVANCIA: Un ítem es relevante si cumple en cierta medida con las preferencias del usuario (habría que ajustar como medimos si cumple o no)

PARA VERIFICAR LA NOVEDAD: Fijarse nomás si el ítem recomendado ya tiene rating por parte del usuario (si ya lo tiene, está mal, no cumple)

PARA VERIFICAR LA DIVERSIDAD: Mepa que hay métricas de diversidad hechas para esto, utilizaremos esta, más adelante vemos.

### Actividad 1.4: Elaborar un plan de proyecto
Hasta ahora planteamos los objetivos de negocio (lo que quieren los stakeholders), analizamos la situación actual y determinamos qué queremos lograr en el proceso de minería de datos. A partir de ahora, el plan es el siguiente:
1. **Entender los datos**: Coincide con la fase 2 de CRISP-DM. Buscamos:
	* Familiarizarnos con el dataset.
	* Analizar cada atributo (tipo, semántica, rango).
	* Detectar anomalías
	* Verificar la calidad de los datos.
2. **Preparar los datos para el modelo**: Coincide con la fase 3 de CRISP-DM. Buscamos:
	* Asegurar la calidad de los datos.
	* Preparar esos datos para el modelado.
3. **Modelar el sistema recomendador**: Coincide con la fase 4 de CRISP-DM. Buscamos:
	* Seleccionar la técnica de modelado adecuada basándonos en la información obtenida en todo el proceso anterior.
	* Definir métricas de evaluación de los sistemas recomendadores candidatos.
	* Construir los sistemas recomendadores.
	* Evaluarlos según los criterios definidos anteriormente.
	* Elegir el sistema recomendador que más se ajuste a esos criterios.
4. **Evaluar el modelo**: Coincide con la fase 5 de CRISP-DM. Tenemos que verificar que el sistema recomendador elegido cumpla con los objetivos de la minería de datos planteados en la fase 1.
5. **Desplegar el modelo**: Coincide con la fase 6 de CRISP-DM. Consiste en definir cómo implementaremos el sistema recomendador y definir un plan de seguimiento del mismo.

Cabe aclarar que en todo momento tendremos en cuenta que CRISP-DM es una metodología **iterativa**, y por ende contempla (y casi que exige) que no se siga una secuencia lineal entre fases. A lo largo del proceso volveremos hacia atrás entre fases y actividades siempre que lo consideremos necesario. El plan solo representa una secuencia ideal de trabajo, que puede ser ajustada o modificada durante el desarrollo del proyecto.

## Fase 2 - Entendimiento de los datos
El objetivo principal de esta fase es familiarizarnos con los datos. Antes de empezar con este proceso, es necesario cargar todas las librerías que utilizaremos a lo largo del trabajo.

In [42]:
import pandas as pd
import ast

### Actividad 2.1: Recolección inicial de los de datos
Los datos de los 100 juegos se encuentran en el archivo subido a GitHub database/games.csv, así que procedemos a cargar este archivo en nuestro Jupyter y verificamos si se cargó correctamente

In [43]:
df_games = pd.read_csv("database/games.csv")

df_games.head(3)

,id,title,releaseDate,rating,genres,description,platforms,metascore,metascore_count,metascore_sentiment,userscore,userscore_count,userscore_sentiment,platform_metascores,developer,publisher,platform_list,genre_list
0,1300540639,Outer Wilds: Echoes of the Eye,2021-09-28,NaN,"Open-World Action, Action RPG, Linear Action A...",A strange satellite photo that can’t be explai...,PC,82.0,8,Generally favorable,89,204,Generally favorable,82,"Mobius Digital, LLC",Annapurna Interactive,['PC'],"['Open-World Action', 'Action RPG', 'Linear Ac..."
1,1300473095,Yakuza Kiwami 2,2018-08-28,M,"Open-World Action, FPS, Survival",Kazuma Kiryu thought his Tojo Clan days were b...,"PlayStation 4,PC,Xbox One,Nintendo Switch 2",85.0,69,Generally favorable,84,688,Generally favorable,"85,82,88,83",Ryu ga Gotoku Studios,Sega,"['PlayStation 4', 'PC', 'Xbox One', 'Nintendo ...","['Open-World Action', 'FPS', 'Survival']"
2,1300536274,Lost Judgment,2021-09-21,M,"Open-World Action, FPS, Survival","SEIZE THE TRUTH - December 2021, Tokyo distric...","PlayStation 5,Xbox Series X,PlayStation 4,PC",82.0,80,Generally favorable,85,416,Generally favorable,"82,80,83,84",Ryu ga Gotoku Studios,Sega,"['PlayStation 5', 'Xbox Series X', 'PlayStatio...","['Open-World Action', 'FPS', 'Survival']"


Estos datos fueron extraídos ..... ACÁ HAY QUE METER UNA JUSTIFICACIÓN        "En esta fase hay que obtener los datos que serán utilizados para el proyecto y documentar adecuadamente su origen"

### Actividad 2.2: Descripción de los datos
Necesitamos realizar una descripción detallada de los datos y de sus particularidades.

#### Instancia 2.2.1: Tamaño del dataset y atributos
Empezaremos revisando el tamaño del dataset para corroborar que sí tiene 100 juegos en total.

In [44]:
df_games.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   100 non-null    int64  
 1   title                100 non-null    str    
 2   releaseDate          100 non-null    str    
 3   rating               78 non-null     str    
 4   genres               100 non-null    str    
 5   description          100 non-null    str    
 6   platforms            100 non-null    str    
 7   metascore            100 non-null    float64
 8   metascore_count      100 non-null    int64  
 9   metascore_sentiment  100 non-null    str    
 10  userscore            100 non-null    int64  
 11  userscore_count      100 non-null    int64  
 12  userscore_sentiment  100 non-null    str    
 13  platform_metascores  100 non-null    str    
 14  developer            100 non-null    str    
 15  publisher            100 non-null    str    
 16  pl

Efectivamente, el dataset cuenta con 100 elementos.

Además, es conveniente listar los atributos que tiene cada juego:
* id
* title
* releaseDate
* rating
* genres
* description
* platforms
* metascore
* metascore_count
* metascore_sentiment
* userscore
* userscore_count
* userscore_sentiment
* platform_metascores
* developer
* publisher
* platform_list
* genre_list

#### Instancia 2.2.2: Revisión de valores nulos
Con df_games.info() también obtenemos información extra de utilidad. Podemos ver que solo una columna tiene valores nulos (22), y es la columna "rating". Dejamos pendiente el análisis de estos valores nulos para la actividad 2.4 (verificación de la calidad de los datos).

#### Instancia 2.2.3: Análisis del atributo "rating"

Ahora, queremos describir c/atributo y entenderlo. Queremos conocer:
* Su tipo
* Su rango de valores
* Su semántica asociada
* Tener una idea de su distribución estadística

Y ya que estabamos hablando del atributo "rating", empezaremos con este.

In [45]:
print(df_games["rating"].dtype)
df_games["rating"].value_counts()

str


rating
M       49
T       26
E10+     2
E        1
Name: count, dtype: int64

Los valores posibles son:
* M
* T
* E10+
* E

Inicialmente pensabamos que estaba relacionado a la puntuación del juego, pero no tiene mucho sentido que esté codificada como string. Investigando qué significan estas letras, encontramos que corresponden al sistema de clasificación por edades de ESRB. Estas clasificaciones se muestran en la página oficial de ESRB.

[Guía oficial de clasificaciones ESRB](https://www.esrb.org/ratings-guide/)

Significado de c/valor:
* M (Mature) → 17+ años. Contenido para adultos, puede contener violencia fuerte, lenguaje fuerte, etc.
* T (Teen) → 13+ años. Contenido para adolescentes o adultos, puede haber violencia moderada, lenguaje leve o temas sugestivos.
* E10+ (Everyone 10+) → 10+ años. Algo más de acción o violencia caricaturesca.
* E (Everyone) → Apto para todo público.

Este atributo es de tipo ordinal. Sus valores representan categorías de clasificación etaria de los videojuegos. Las categorías no son cuantitativas, pero poseen un orden natural según la edad del usuario. Se trata de categorías, como un atributo nominal, pero al tener un orden, es un atributo ordinal.

Al ser un atributo ordinal, el dominio de "rating" serían las categorías de ESRB ya mencionadas, que ordenados de menor a mayor serían:
1. E
2. E10+
3. T
4. M

En consecuencia, el rango ordinal del atributo va desde E (menor restricción) hasta M (mayor restricción).

En cuánto a la distribución estadística, podemos ver que la mayoría de juegos son para adultos, otra porción considerable son para adolescentes, y casi ninguno de ellos son para mayores de 10 o para todo público. No consideramos necesario utilizar descriptores básicos, son pocas categorías y su distribución estadística es observable simplemente con la cantidad de datos por categoría.

**Conclusiones del atributo "rating":**
* Tipo: Ordinal.
* Semántica: Edad del usuario recomendada para el juego.
* Dominio (ordenado): [E, E10+, T, M].
* Distribución estadística: La mayoría son juegos para adultos o adolescentes.

#### Instancia 2.2.4: Análisis del atributo "releaseDate"
Traduciendolo al español, sabemos que la semántica de este atributo es la fecha de lanzamiento del juego (salvo que durante su descripción descubramos algo diferente a esto, mantendremos ésta interpretación). Veamos algunos valores y observemos cómo está codificado el atributo.

In [53]:
df_games["releaseDate"].dtype
df_games["releaseDate"].head(5)

0    2021-09-28
1    2018-08-28
2    2021-09-21
3    2012-08-14
4    2019-10-08
Name: releaseDate, dtype: str

Las fechas están en formato string, pero el tipo de atributo es numérico intervalado-discreto. Es numérico porque es un atributo cuantitativo, es discreto porque entre dos valores no siempre hay un valor (por ejemplo, dos fechas correspondientes a días consecutivos), y es intervalado porque los valores están en una escala ordenada donde las diferencias entre valores tienen significado (podemos calcular cuántos días hay entre dos fechas).

Para conocer el rango, pasaremos las fechas codificadas como string a formato date, verificamos que no se haya perdido ninguna fecha por el camino y miramos los máximos y mínimos

In [54]:
df_games_datesfixed = df_games.copy()
df_games_datesfixed["releaseDate"] = pd.to_datetime(df_games["releaseDate"], errors="coerce")
print(len(df_games_datesfixed))
df_games_datesfixed["releaseDate"].agg(["min","max"])

100


min   1999-06-17
max   2025-11-11
Name: releaseDate, dtype: datetime64[us]

El rango de fechas entonces es [1996-06-17, 2025-11-11]

Para darnos una idea de la distribución estadística, utilizaremos descriptores estadísticos. Para ello, nos conviene discretizar las fechas en años, y ahí calculamos la moda, la mediana y el promedio:

In [55]:
df_games_dates_as_year = df_games_datesfixed.copy()
df_games_dates_as_year["releaseDate"] = pd.to_datetime(df_games_datesfixed["releaseDate"], errors="coerce").dt.year

# Verifico que el cambio se haya hecho correctamente
print(len(df_games_dates_as_year))
print("Min:", df_games_dates_as_year["releaseDate"].min())
print("Max:", df_games_dates_as_year["releaseDate"].max())

# Calculo el promedio, mediana y moda
promedio = df_games_dates_as_year["releaseDate"].mean()
mediana = df_games_dates_as_year["releaseDate"].median()
moda = df_games_dates_as_year["releaseDate"].mode()[0]
cantidad = df_games_dates_as_year["releaseDate"].value_counts().iloc[0]

print(f"Promedio: {promedio}")
print(f"Mediana: {mediana}")
print(f"Moda: {moda}, cantidad de juegos en ese año: {cantidad}")

100
Min: 1999
Max: 2025
Promedio: 2015.56
Mediana: 2017.0
Moda: 2024, cantidad de juegos en ese año: 12


La distribución parece bastante equilibrada dentro del rango analizado. Sí, está sesgada hacia la izquierda, pero menos de lo que se esperaba. Se podía esperar que, como en los últimos años se han lanzado más juegos que nunca, el promedio, la mediana y la moda iban a estar más sesgados hacia la izquierda, y no es el caso. Aún así, como Promedio (2015.56) < Mediana (2017.0) < Moda (2024), podemos afirmar que tiene un sesgo negativo (hacia la izquierda). Otra observación es que podíamos pensar que el juego del 1996 era una "anomalía", pero en cierto modo, que el sesgo hacia los últimos años no sea excesivo, nos discipa de dudas (por el momento).

**Conclusiones del atributo "releaseDate":**
* Tipo: Numérico intervalado-discreto.
* Semántica: Fecha de lanzamiento del juego.
* Rango: [1996-06-17, 2025-11-11].
* Distribución estadística: Pequeño sesgo hacia la izquierda (sesgo negativo).

#### Instancia 2.2.5: Análisis del atributo "genres" y "genres_list"
"genres", por lo observado al ejecutar df_games.info(), contiene los géneros a los que pertenece el juego separados por coma (esa es la semántica del atributo). Veamos cuáles son los géneros posibles y de paso cuántos juegos pertenecen a cada género.

In [64]:
df_games["genres"].str.split(", ").explode().value_counts()

genres
Open-World Action          89
FPS                        69
Action RPG                 67
Linear Action Adventure    41
Survival                   34
Name: count, dtype: int64

Entonces, el juego puede pertenecer a uno o más de estos géneros:
* Open-World Action.
* FPS.
* Action RPG.
* Linear Action Adventure.
* Survival.

Este listado de valores posibles es el dominio del atributo. Es un atributo de tipo nominal, pues está formado por categorías que no tienen un orden específico ni distancia entre ellas. Además es multivaluado, puede tomar varios valores simultáneamente.

Estadísticamente:
* El género Open-World Action aparece en la mayoría de juegos (89).
* Más de la mitad de los juegos incluyen los géneros FPS (69) o Action RPG (67).
* Aproximadamente un tercio de los juegos presentan los géneros Linear Action Adventure (41) o Survival (34).

El atributo "genres" y el atributo "genres_list" significan lo mismo. Ambos contienen los géneros a los que pertenece el juego, solo cambia su codificación. Lo podemos verificar en el siguiente código

In [65]:
df_games[["genre_list", "genres"]].head()

,genre_list,genres
0,"['Open-World Action', 'Action RPG', 'Linear Ac...","Open-World Action, Action RPG, Linear Action A..."
1,"['Open-World Action', 'FPS', 'Survival']","Open-World Action, FPS, Survival"
2,"['Open-World Action', 'FPS', 'Survival']","Open-World Action, FPS, Survival"
3,"['Open-World Action', 'Linear Action Adventure...","Open-World Action, Linear Action Adventure, FPS"
4,"['Open-World Action', 'Linear Action Adventure...","Open-World Action, Linear Action Adventure, FPS"


Nos encargaremos de tratar esta redundancia en la fase 3 (preparación de los datos).

**Conclusiones del atributo "genres":**
* Tipo: Nominal.
* Semántica: Géneros a los que pertenece el juego.
* Dominio: [Open-World Action, FPS, Action RPG, Linear Action Adventure, Survival].
* Distribución estadística: Muchísimos Open-World Action, bastantes FPS y Action RPG, y una cantidad considerable de Linear Action Adventure o Survival. 

### Actividad 2.3: Exploración de los datos
BLA

### Actividad 2.4: Verificación de la calidad de los datos
BLA

#### Instancia 2.4.X: Análisis de valores nulos
Recapitulando, teníamos 22 valores nulos en el atributo "rating". Los nulos pueden tener varias explicaciones lógicas. Pueden ser porque:
* El juego es apto para todo público y no se especificó porque no tiene restricciones.
* El juego no fue clasificado por ESRB.
* El juego si tiene clasificación pero el dataset no la registró porque la clasificación se realizó hace poco y el juego se cargó hace mucho en el dataset.

Esta última explicación hace bastante sentido, así que indagaremos un poco más. Disponemos de la fecha de lanzamiento del juego, así que como también contamos con el atributo "releaseDate", o fecha de lanzamiento en español, podemos comparar los datos que tienen nulo en "rating" y los que no tienen nulo para ver si existe un sesgo en su fecha de lanzamiento y corroborar si la fecha es un justificativo de los valores nulos.

## Fase 3 - Preparación de los datos
BLA

### Actividad 3.1: Selección de los datos
BLA

### Actividad 3.2: Limpieza de datos
BLA

### Actividad 3.3: Integración de los datos
BLA

### Actividad 3.4: Reducción de los datos
BLA

### Actividad 3.5: Transformación de los datos
BLA

## Fase 4 - Modelado
BLA

### Actividad 4.1: Selección de técnicas de modelado
BLA

### Actividad 4.2: Diseño del test del modelo
BLA

### Actividad 4.3: Construcción de modelos
BLA

### Actividad 4.4: Evaluación del modelo
BLA

## Fase 5 - Evaluación
BLA

### Actividad 5.1: Evaluación de los resultados del modelo
BLA

### Actividad 5.2: Revisión del proceso
BLA

### Actividad 5.3: Determinación de próximos pasos
BLA

## Fase 6 - Despliegue
BLA

### Actividad 6.1: Plan de despliegue
BLA

### Actividad 6.2: Plan de monitoreo y mantenimiento
BLA

### Actividad 6.3: Producción del informe final
BLA

### Actividad 6.4: Presentación de resultados
BLA